# guppy and hugr to qir conversion and submission to H2


This example shows how to convert guppy to qir which can be submitted directly to H1 and H2 device, emulator and syntax checker.

You need to install hugr-qir, guppy and pytket-quantinuum for this notebook to work.

### Current guppy features that can't be converted:
- loops with condition not known at compiletime
- functions returning qubit arrays
- RNG functions
- dynamic qubit allocation


In [ ]:
# You can write your guppy directly in a notebook or in a separate file
from typing import no_type_check

from guppylang import guppy, qubit
from guppylang.std.builtins import result
from guppylang.std.quantum import h, measure


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    h(q0)
    h(q1)

    b0 = measure(q0)
    b1 = measure(q1)
    b2 = b0 ^ b1

    result("0", b2)

# Convert hugr to qir

By default, the function will automatically check the generated QIR to capture most of the issues that could happen.
This will show an error message with more details about the problem the check can be turned off using the keyword argument `validate_qir = False`

In [ ]:
from hugr_qir.guppy_to_qir import guppy_to_qir_str, guppy_to_qir_bytes
guppy_qir_bitcode_string = guppy_to_qir_str(main)

In [ ]:
# To get a human-readable LLVM assemly language string use the `hugr_to_qir` function with the keyword argument `output_format = OutputFormat.LLVM_IR`
from hugr_qir.guppy_to_qir import guppy_to_qir_str, guppy_to_qir_bytes

guppy_qir = guppy_to_qir_str(main)
print(guppy_qir)

### Loops in the program are unrolled automatically when possible, because backwards branching is not available on H Series.  This means that the QIR generated from programs containing loops can get quite long as shown here:

In [ ]:
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    for _ in range(10):
        q3 = qubit()
        h(q3)
        b = measure(q3)
        if b:
            h(q0)

    result("0", measure(q0))
    result("1", measure(q1))

In [ ]:
guppy_qir = guppy_to_qir_str(main, validate_qir=False)
print(guppy_qir)

### A similar example to the one above with a loop that terminates based on a measurement result inside the loop leads to the generation of QIR which is valid within the standard, but can't be executed on H-Series devices.

In [ ]:
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    i = 0
    while i < 10:
        q3 = qubit()
        h(q3)
        b = measure(q3)
        if b:
            h(q0)
            i += 1

    result("0", measure(q0))
    result("1", measure(q1))

In [ ]:
guppy_qir = guppy_to_qir_str(main, validate_qir=False)
print(guppy_qir)

In [ ]:
# this will fail because of the loop in the generated QIR

try:
    guppy_qir = to_qir_str(main)
    print(guppy_qir)
except Exception as e:
    print("Validation failed as expected:")
    print(e)

# Submission to the device via Nexus

The QIR generated can be submitted directly to Nexus. The python Nexus API is available via `pip install qnexus`. This requires a different QIR format for the submission, which can be generated from `compile_qir`.

In [ ]:
import qnexus as qnx

qnx.login()

In [ ]:
import datetime

project = qnx.projects.get_or_create(name="QIR-Demonstration3")
qnx.context.set_active_project(project)

qir_name = "HUGR-QIR"
jobname_suffix = datetime.datetime.now().strftime("%Y_%m_%d-%H-%M-%S")

In [ ]:
# You can write your guppy directly in a notebook or in a separate file
from typing import no_type_check

from guppylang import guppy


@guppy
@no_type_check
def main() -> None:
    q0 = qubit()
    q1 = qubit()

    h(q0)
    h(q1)

    b0 = measure(q0)
    b1 = measure(q1)
    b2 = b0 ^ b1

    result("0", b2)

In [ ]:
guppy_qir_bitcode = guppy_to_qir_bytes(main)

In [ ]:
qir_program_ref = qnx.qir.upload(qir=guppy_qir_bitcode, name=qir_name, project=project)

In [ ]:
# Run on the H2-1 Syntax checker
device_name = "H2-1SC"

qnx.context.set_active_project(project)
config = qnx.QuantinuumConfig(device_name=device_name)

job_name = f"execution-job-qir-{qir_name}-{device_name}-{jobname_suffix}"
ref_execute_job = qnx.start_execute_job(
    programs=[qir_program_ref],
    n_shots=[10],
    backend_config=config,
    name=job_name,
)

In [ ]:
qnx.jobs.wait_for(ref_execute_job)

In [ ]:
qir_result = qnx.jobs.results(ref_execute_job)[0].download_result()
qir_result.get_counts()